# Track A Provider Comparison (Colab Runner)

Run this notebook in Colab Web to execute full Track A benchmarks (style + genre), then generate slide-ready summary tables.

## 0) Runtime

- Runtime -> Change runtime type -> GPU
- If available, use a stronger runtime (A100/high-RAM) for faster end-to-end runs

In [ ]:
# 0.5) Create a complete Track A workspace in Colab root (/content)
from pathlib import Path
import os

WORK_DIR = Path('/content/tracka_workspace')
folders = [
    WORK_DIR,
    WORK_DIR / 'app',
    WORK_DIR / 'scripts',
    WORK_DIR / 'configs',
    WORK_DIR / 'artifacts',
    WORK_DIR / 'artifacts' / 'runs',
    WORK_DIR / 'logs',
    WORK_DIR / 'notebooks',
]
for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)

# Create placeholder files so upload targets are obvious in Colab file browser
placeholder_files = [
    WORK_DIR / 'app' / '__init__.py',
    WORK_DIR / 'app' / 'experiment_models.py',
    WORK_DIR / 'scripts' / 'run_track_a_comparison.py',
    WORK_DIR / 'scripts' / 'benchmark_wikiart_zero_shot.py',
    WORK_DIR / 'scripts' / 'merge_track_a_runs.py',
    WORK_DIR / 'configs' / 'zero_shot_models.json',
    WORK_DIR / 'requirements.txt',
]
for file_path in placeholder_files:
    file_path.touch(exist_ok=True)

os.chdir(WORK_DIR)

print('Workspace ready at:', WORK_DIR)
print('\nUpload/replace these files in place:')
print('  app/__init__.py')
print('  app/experiment_models.py')
print('  scripts/run_track_a_comparison.py')
print('  scripts/benchmark_wikiart_zero_shot.py')
print('  scripts/merge_track_a_runs.py')
print('  configs/zero_shot_models.json')
print('  requirements.txt')
print('\nRaw WikiArt parquet upload is NOT required (streaming mode is used).')
print('Current CWD:', Path.cwd())

In [ ]:
# 1) Mount Drive (optional but recommended for saving outputs)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2) Set project path
# Default: use the workspace created in Cell 0.5
import os
from pathlib import Path

PROJECT_DIR = Path('/content/tracka_workspace')
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_DIR)
print('CWD:', os.getcwd())

In [ ]:
# 3) Install dependencies
# sentencepiece + protobuf are required for SigLIP tokenizers
%pip install -q -r requirements.txt sentencepiece protobuf

In [ ]:
# 4) (Optional) Set Hugging Face token for faster/reliable downloads
import os
from getpass import getpass
tok = getpass('HF_TOKEN (optional): ').strip()
if tok:
    os.environ['HF_TOKEN'] = tok
    print('HF_TOKEN set for this runtime session')
else:
    print('No HF token set; continuing unauthenticated')

In [ ]:
# 5) Preflight checks (required files; raw parquet upload not required in streaming mode)
from pathlib import Path
required = [
    'scripts/run_track_a_comparison.py',
    'scripts/benchmark_wikiart_zero_shot.py',
    'scripts/merge_track_a_runs.py',
    'configs/zero_shot_models.json',
]
missing = [p for p in required if not Path(p).exists()]
print('Missing required files:', missing if missing else 'None')

USE_STREAMING = True
STREAMING_REPO = 'huggan/wikiart'
STREAMING_SPLIT = 'train'
print('USE_STREAMING:', USE_STREAMING)
print('STREAMING_REPO:', STREAMING_REPO, '| split:', STREAMING_SPLIT)

if missing:
    raise FileNotFoundError('Add missing files before running Track A.')

In [ ]:
# 6) Smoke run (quick validation; ~32 images) in streaming mode
!python scripts/run_track_a_comparison.py \
  --models-config configs/zero_shot_models.json \
  --output-root artifacts/runs \
  --targets style genre \
  --templates-per-class 1 \
  --max-images 32 \
  --streaming-repo-id huggan/wikiart \
  --streaming-split train

In [ ]:
# 7) Full Track A run (style + genre) in streaming mode
!python scripts/run_track_a_comparison.py \
  --models-config configs/zero_shot_models.json \
  --output-root artifacts/runs \
  --targets style genre \
  --templates-per-class 4 \
  --max-images 0 \
  --streaming-repo-id huggan/wikiart \
  --streaming-split train

In [ ]:
# 8) Merge all Track A runs into final slide-ready tables
!python scripts/merge_track_a_runs.py \
  --runs-root artifacts/runs \
  --output-dir artifacts/runs/trackA_merged

In [ ]:
# 9) Show where outputs are
from pathlib import Path
runs = sorted(Path('artifacts/runs').glob('trackA_*'))
print('TrackA run folders:')
for p in runs[-5:]:
    print(' -', p)
print('Merged folder:', Path('artifacts/runs/trackA_merged').resolve())

In [ ]:
# 10) (Optional) Zip merged outputs for download
!cd artifacts/runs && zip -r trackA_merged.zip trackA_merged